# Trading algorítmico — Act 04 a Act 07 en un solo notebook

| Actividad | Contenido |
|---|---|
| **Act 04** | Indicadores técnicos, nulos, gráficas, SMA10 vs SMA100, tramos donde fallan, correlación, look-ahead |
| **Act 05** | `SPEC.md`, `src/strategy.py` (entry rule + posición), unit tests, win rate de break-even |
| **Act 06** | Backtest orientado a eventos, golden file, truncamiento del pipeline, métricas, costos, auditoría de sesgos |
| **Act 07** | Features de régimen, reglas / K-means / HMM, tabla comparativa, Viterbi vs filtrada, θ por régimen |

El notebook **escribe los módulos del repo** (`src/`, `tests/`, `SPEC.md`) con `%%writefile`, así que al correrlo de arriba a abajo queda la estructura lista para subir a GitHub.

> ⚠️ Todos los textos de interpretación que dependen de los datos se generan **con f-strings a partir de tus resultados**. Si cambias de activo o de fechas, léelos y ajusta la redacción con tus propias palabras antes de entregar.

In [ ]:
# !pip install yfinance hmmlearn scikit-learn scipy matplotlib pandas pytest -q
import os, sys, importlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams["figure.dpi"] = 100

for d in ("src", "tests", "data", "figures"):
    os.makedirs(d, exist_ok=True)
if os.path.abspath("src") not in sys.path:
    sys.path.insert(0, os.path.abspath("src"))

# Act 04 — Technical Indicators
## 1. Datos

In [ ]:
%%writefile src/data_utils.py
"""Carga de datos: yfinance si hay internet, si no un OHLCV sintético con regímenes."""
import os
import numpy as np
import pandas as pd


def synthetic_ohlcv(n=3500, seed=42, start="2012-01-03"):
    """Serie OHLCV sintética con 3 regímenes de Markov (alcista, lateral, estrés)."""
    rng = np.random.default_rng(seed)
    P = np.array([[0.985, 0.010, 0.005],
                  [0.020, 0.970, 0.010],
                  [0.030, 0.020, 0.950]])
    mu = np.array([0.0007, 0.0, -0.0015])
    sig = np.array([0.008, 0.011, 0.025])
    s, rets = 0, np.empty(n)
    for t in range(n):
        s = rng.choice(3, p=P[s])
        rets[t] = rng.normal(mu[s], sig[s])
    close = 100 * np.exp(np.cumsum(rets))
    prev = np.r_[100.0, close[:-1]]
    open_ = prev * np.exp(rng.normal(0, 0.002, n))
    spread = np.abs(rng.normal(0, 1, n)) * np.abs(rets).mean()
    high = np.maximum(open_, close) * np.exp(spread)
    low = np.minimum(open_, close) * np.exp(-spread)
    vol = rng.lognormal(17, 0.3, n)
    idx = pd.bdate_range(start, periods=n)
    return pd.DataFrame({"open": open_, "high": high, "low": low,
                         "close": close, "volume": vol}, index=idx)


def load_data(ticker="SPY", start="2012-01-01", end="2025-12-31", folder="data"):
    path = os.path.join(folder, f"{ticker}.csv")
    if os.path.exists(path):
        return pd.read_csv(path, index_col=0, parse_dates=True)
    try:
        import yfinance as yf
        df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df.rename(columns=str.lower)[["open", "high", "low", "close", "volume"]].dropna()
        if len(df) < 500:
            raise ValueError("descarga incompleta")
        os.makedirs(folder, exist_ok=True)
        df.to_csv(path)
        return df
    except Exception as e:  # sin internet / sin yfinance
        print(f"[aviso] No se pudo descargar {ticker} ({e}). Uso datos SINTÉTICOS.")
        return synthetic_ohlcv()

In [ ]:
import data_utils; importlib.reload(data_utils)
TICKER, START, END = "SPY", "2012-01-01", "2025-12-31"
raw = data_utils.load_data(TICKER, START, END)
print(raw.shape, raw.index.min().date(), "→", raw.index.max().date())
raw.tail()

## 2. `indicadores.add_indicators(df)`
Todos los indicadores usan solo ventanas hacia atrás (`rolling`, `ewm(adjust=False)`), por lo que deben pasar la prueba de truncamiento.

In [ ]:
%%writefile src/indicadores.py
"""Indicadores técnicos 100% causales (solo usan información hasta t)."""
import numpy as np
import pandas as pd

# Familia de cada indicador (para el SPEC y para leer la matriz de correlación)
FAMILIES = {
    "sma_10": "tendencia", "sma_20": "tendencia", "sma_50": "tendencia", "sma_100": "tendencia",
    "ema_12": "tendencia", "ema_26": "tendencia",
    "macd": "momentum", "macd_signal": "momentum", "macd_hist": "momentum",
    "rsi_14": "momentum/oscilador", "roc_10": "momentum", "stoch_k": "oscilador", "stoch_d": "oscilador",
    "bb_upper": "volatilidad", "bb_lower": "volatilidad", "bb_pctb": "volatilidad/reversión",
    "atr_14": "volatilidad", "zscore_20": "reversión a la media", "obv": "volumen",
}


def sma(s, n):
    return s.rolling(n, min_periods=n).mean()


def ema(s, n):
    return s.ewm(span=n, adjust=False, min_periods=n).mean()


def wilder(s, n):
    return s.ewm(alpha=1 / n, adjust=False, min_periods=n).mean()


def rsi(close, n=14):
    d = close.diff()
    up, down = d.clip(lower=0), (-d).clip(lower=0)
    rs = wilder(up, n) / wilder(down, n)
    return 100 - 100 / (1 + rs)


def atr(df, n=14):
    pc = df["close"].shift(1)
    tr = pd.concat([df["high"] - df["low"], (df["high"] - pc).abs(), (df["low"] - pc).abs()], axis=1).max(axis=1)
    return wilder(tr, n)


def add_indicators(df):
    """Devuelve una copia de df con todos los indicadores añadidos."""
    out = df.copy()
    c = out["close"]
    for n in (10, 20, 50, 100):
        out[f"sma_{n}"] = sma(c, n)
    out["ema_12"], out["ema_26"] = ema(c, 12), ema(c, 26)
    out["macd"] = out["ema_12"] - out["ema_26"]
    out["macd_signal"] = out["macd"].ewm(span=9, adjust=False, min_periods=9).mean()
    out["macd_hist"] = out["macd"] - out["macd_signal"]
    out["rsi_14"] = rsi(c, 14)
    out["roc_10"] = c.pct_change(10)
    ll, hh = out["low"].rolling(14).min(), out["high"].rolling(14).max()
    out["stoch_k"] = 100 * (c - ll) / (hh - ll)
    out["stoch_d"] = out["stoch_k"].rolling(3).mean()
    mid, sd = sma(c, 20), c.rolling(20, min_periods=20).std(ddof=0)
    out["bb_upper"], out["bb_lower"] = mid + 2 * sd, mid - 2 * sd
    out["bb_pctb"] = (c - out["bb_lower"]) / (out["bb_upper"] - out["bb_lower"])
    out["zscore_20"] = (c - mid) / sd
    out["atr_14"] = atr(out, 14)
    out["obv"] = (np.sign(c.diff()).fillna(0) * out["volume"]).cumsum()
    out["ret_1"] = np.log(c).diff()
    return out

In [ ]:
import indicadores; importlib.reload(indicadores)
df = indicadores.add_indicators(raw)
IND_COLS = list(indicadores.FAMILIES)
df[IND_COLS].describe().T[["mean", "std", "min", "max"]]

## 3. Verificación de valores nulos
Los nulos deben aparecer **solo al inicio** (periodo de calentamiento = ventana del indicador). Si hubiera nulos en medio de la serie serían huecos en los datos.

In [ ]:
nulls = pd.DataFrame({"nulos": df[IND_COLS].isna().sum(),
                      "primer_valido": df[IND_COLS].apply(lambda s: s.first_valid_index().date())})
nulls["nulos_despues_del_warmup"] = [df[c].loc[df[c].first_valid_index():].isna().sum() for c in IND_COLS]
display(nulls)
assert nulls["nulos_despues_del_warmup"].sum() == 0, "Hay huecos a mitad de la serie"
WARMUP = int(nulls["nulos"].max())
print(f"Warm-up máximo: {WARMUP} barras (sma_100). Se descartan antes de modelar.")

## 4. Visualización con `src/plots.py`

In [ ]:
%%writefile src/plots.py
import matplotlib.pyplot as plt


def _cut(df, start=None, end=None):
    return df.loc[start:end]


def plot_overlays(df, cols=("sma_10", "sma_100"), start=None, end=None, ax=None, bands=False, title=None):
    d = _cut(df, start, end)
    ax = ax or plt.subplots(figsize=(12, 4))[1]
    ax.plot(d.index, d["close"], color="black", lw=1, label="close")
    for c in cols:
        ax.plot(d.index, d[c], lw=1.2, label=c)
    if bands:
        ax.fill_between(d.index, d["bb_lower"], d["bb_upper"], alpha=0.12, label="Bollinger 20,2")
    ax.legend(loc="upper left", fontsize=8); ax.set_title(title or "Precio + overlays"); ax.grid(alpha=.3)
    return ax


def plot_rsi(df, start=None, end=None, ax=None, lo=30, hi=70):
    d = _cut(df, start, end)
    ax = ax or plt.subplots(figsize=(12, 2.5))[1]
    ax.plot(d.index, d["rsi_14"], color="purple", lw=1)
    ax.axhline(hi, ls="--", c="r", lw=.8); ax.axhline(lo, ls="--", c="g", lw=.8)
    ax.set_ylim(0, 100); ax.set_title("RSI(14)"); ax.grid(alpha=.3)
    return ax


def plot_macd(df, start=None, end=None, ax=None):
    d = _cut(df, start, end)
    ax = ax or plt.subplots(figsize=(12, 2.5))[1]
    ax.plot(d.index, d["macd"], lw=1, label="MACD")
    ax.plot(d.index, d["macd_signal"], lw=1, label="señal")
    ax.bar(d.index, d["macd_hist"], color=["g" if v > 0 else "r" for v in d["macd_hist"].fillna(0)], alpha=.4, width=1)
    ax.axhline(0, c="k", lw=.6); ax.legend(fontsize=8, loc="upper left"); ax.set_title("MACD(12,26,9)"); ax.grid(alpha=.3)
    return ax


def plot_panel(df, start=None, end=None, overlays=("sma_20", "sma_100"), savepath=None):
    fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1.2, 1.2, 1]})
    plot_overlays(df, overlays, start, end, ax=axes[0], bands=True, title="Panel completo")
    plot_rsi(df, start, end, ax=axes[1])
    plot_macd(df, start, end, ax=axes[2])
    d = _cut(df, start, end)
    axes[3].plot(d.index, d["atr_14"], c="brown", lw=1); axes[3].set_title("ATR(14)"); axes[3].grid(alpha=.3)
    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, dpi=120)
    return fig

In [ ]:
import plots; importlib.reload(plots)
last2y = df.index[-504]
plots.plot_overlays(df, ("sma_20", "sma_50", "sma_100"), start=last2y, bands=True, title="Overlays sobre el precio (últimos 2 años)")
plt.show()
plots.plot_rsi(df, start=last2y); plt.show()
plots.plot_macd(df, start=last2y); plt.show()
plots.plot_panel(df, start=last2y, savepath="figures/panel_completo.png"); plt.show()

## 5. Preguntas
### 5.1 `sma_10` vs `sma_100` sobre el mismo precio

In [ ]:
plots.plot_overlays(df, ("sma_10", "sma_100"), start=last2y, title="SMA10 vs SMA100")
plt.show()

def cross_stats(price, ma, n, whipsaw=10, horizon=20):
    valid = ma.notna()
    above = (price > ma)
    cross = (above != above.shift()) & valid & valid.shift(fill_value=False)
    idx = np.flatnonzero(cross.values)
    gaps = np.diff(idx)
    fwd = (price.shift(-horizon) / price - 1)
    up = cross & above
    years = valid.sum() / 252
    return pd.Series({"lag teórico (barras)": (n - 1) / 2,
                      "cruces totales": len(idx),
                      "cruces por año": len(idx) / years,
                      "barras promedio entre cruces": gaps.mean(),
                      "% cruces revertidos en ≤10 barras (falsos)": (gaps <= whipsaw).mean(),
                      "ret. medio 20d tras cruce alcista": fwd[up].mean(),
                      "% cruces alcistas con ret 20d < 0": (fwd[up] < 0).mean()})

q1 = pd.DataFrame({"sma_10": cross_stats(df.close, df.sma_10, 10), "sma_100": cross_stats(df.close, df.sma_100, 100)})
q1

In [ ]:
a, b = q1["sma_10"], q1["sma_100"]
print(f'''Respuesta:
• Reacciona antes la SMA10: su retraso teórico es {a.iloc[0]:.1f} barras contra {b.iloc[0]:.1f} de la SMA100.
• La SMA10 te habría metido en muchos más movimientos falsos: {a["cruces totales"]:.0f} cruces
  ({a["cruces por año"]:.1f}/año) y {a.iloc[4]:.0%} se revierten en ≤10 barras; la SMA100 solo
  {b["cruces totales"]:.0f} cruces ({b["cruces por año"]:.1f}/año, {b.iloc[4]:.0%} falsos).
• SMA10 → ganas velocidad (entras/sales cerca del giro) y pierdes robustez: más ruido, más
  whipsaws y más costos de transacción.
• SMA100 → ganas filtro de ruido y menos operaciones; pierdes reacción: llega tarde a los giros
  y regala una parte grande del movimiento (y en caídas rápidas sales tarde).''')

### 5.2 Un tramo concreto (con fechas) donde cada indicador se equivoca
Defino para cada indicador su señal "de compra" típica y busco el evento cuyo retorno posterior fue el **peor**. El tramo sale de *tus* datos, así que no coincide con el del profesor.

In [ ]:
def worst_event(event, horizon, side=1):
    fwd = (df.close.shift(-horizon) / df.close - 1) * side
    cand = fwd[event].dropna()
    t0 = cand.idxmin()
    t1 = df.index[df.index.get_loc(t0) + horizon]
    return t0, t1, cand.min()

above10, above100 = df.close > df.sma_10, df.close > df.sma_100
events = {
    "sma_10  (precio cruza arriba)": (above10 & ~above10.shift(fill_value=True), 10),
    "sma_100 (precio cruza arriba)": (above100 & ~above100.shift(fill_value=True) & df.sma_100.notna(), 40),
    "rsi_14  (cruza abajo de 30 → 'sobreventa')": ((df.rsi_14 < 30) & (df.rsi_14.shift() >= 30), 20),
    "macd    (cruce alcista MACD > señal)": ((df.macd > df.macd_signal) & (df.macd.shift() <= df.macd_signal.shift()), 20),
    "bollinger (cierre bajo banda inferior)": ((df.close < df.bb_lower) & (df.close.shift() >= df.bb_lower.shift()), 20),
}
fails = pd.DataFrame([(k, *worst_event(e, h), h) for k, (e, h) in events.items()],
                     columns=["indicador / señal", "inicio", "fin", "retorno posterior", "horizonte"])
display(fails)

fig, axes = plt.subplots(len(fails), 1, figsize=(12, 3.2 * len(fails)))
cols_map = [("sma_10",), ("sma_100",), ("sma_20",), ("ema_12", "ema_26"), ("sma_20",)]
for ax, (_, row), cols in zip(axes, fails.iterrows(), cols_map):
    s = df.index[max(0, df.index.get_loc(row.inicio) - 40)]
    e = df.index[min(len(df) - 1, df.index.get_loc(row.fin) + 20)]
    plots.plot_overlays(df, cols, s, e, ax=ax, bands="bollinger" in row.iloc[0],
                        title=f"{row.iloc[0]}: señal {row.inicio.date()} → {row['retorno posterior']:.1%} a {row.fin.date()}")
    ax.axvspan(row.inicio, row.fin, color="red", alpha=.12)
fig.tight_layout(); fig.savefig("figures/tramos_error.png"); plt.show()

In [ ]:
for _, r in fails.iterrows():
    print(f"• {r.iloc[0].split('(')[0].strip()}: dio señal de compra el {r.inicio.date()} y en {r.horizonte} barras "
          f"(hasta {r.fin.date()}) el precio cambió {r['retorno posterior']:.1%}.")
print('''
Por qué fallan: las medias y el MACD son seguidores de tendencia → fallan en rebotes dentro de
una caída (bull trap) o en mercados laterales. RSI y Bollinger son de reversión → fallan cuando la
caída tiene momentum ("sobrevendido puede volverse más sobrevendido").''')

### 5.3 ¿Qué tan parecidos son entre sí? — matriz de correlación
Uso **Spearman** (robusta a outliers y a relaciones monótonas no lineales). Para no confundir tendencia común del precio, las medias se expresan como distancia relativa al precio.

In [ ]:
X = pd.DataFrame(index=df.index)
for c in ["sma_10", "sma_20", "sma_50", "sma_100", "ema_12", "ema_26", "bb_upper", "bb_lower"]:
    X[c] = df.close / df[c] - 1                     # nivel -> distancia al precio
X["atr_14"] = df.atr_14 / df.close
for c in ["macd", "macd_signal", "macd_hist"]:
    X[c] = df[c] / df.close
for c in ["rsi_14", "roc_10", "stoch_k", "stoch_d", "bb_pctb", "zscore_20"]:
    X[c] = df[c]
X["obv"] = df.obv.diff(20)
corr = X.dropna().corr(method="spearman")

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=8)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iat[i, j]:.1f}", ha="center", va="center", fontsize=6)
plt.colorbar(im); ax.set_title("Correlación de Spearman entre indicadores"); fig.tight_layout()
fig.savefig("figures/corr_indicadores.png"); plt.show()

pairs = corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack().sort_values(key=abs, ascending=False)
print("Pares más redundantes (|ρ| > 0.85):")
display(pairs[pairs.abs() > 0.85].to_frame("rho").head(20))

In [ ]:
n_red = (pairs.abs() > 0.85).sum()
print(f'''Respuesta: hay {n_red} pares con |ρ| > 0.85. Muchos indicadores de "familias distintas" miden casi
lo mismo: RSI, estocástico, %B, z-score y la distancia a SMA10/20 son transformaciones del mismo
momentum de corto plazo. Combinar indicadores muy correlacionados NO agrega información (solo
doble-cuenta la misma señal). Para la estrategia conviene tomar uno por familia poco correlacionada:
tendencia larga (SMA20 vs SMA100), momentum (MACD hist) y un filtro de sobreextensión (RSI).''')

### 5.4 ¿Alguno mira al futuro? — prueba de truncamiento
Para cada `t` muestreado se recalcula `add_indicators(df.iloc[:t+1])` y se compara el valor en `t` contra el de la muestra completa. Si cambia ⇒ look-ahead.

In [ ]:
%%writefile src/lookahead.py
import numpy as np
import pandas as pd


def truncation_test(func, df, cols=None, n_checks=40, warmup=260, seed=0, atol=1e-8):
    """Recalcula func sobre df.iloc[:t+1] y compara la fila t contra la muestra completa.
    Si algún valor cambia -> look-ahead."""
    full = func(df)
    if isinstance(full, pd.Series):
        full = full.to_frame()
    cols = list(cols or full.columns)
    rng = np.random.default_rng(seed)
    ts = sorted(set(rng.choice(np.arange(warmup, len(df)), size=min(n_checks, len(df) - warmup), replace=False)) | {len(df) - 1})
    rows = []
    for t in ts:
        part = func(df.iloc[:t + 1])
        if isinstance(part, pd.Series):
            part = part.to_frame()
        lbl = df.index[t]                       # se compara por fecha (la salida puede estar recortada)
        a = part.reindex([lbl])[cols].astype(float).values[0]
        b = full.reindex([lbl])[cols].astype(float).values[0]
        same = np.isclose(a, b, atol=atol, equal_nan=True)
        rows.append(pd.Series(np.where(same, 0.0, np.abs(np.nan_to_num(a - b, nan=np.inf))), index=cols, name=t))
    diff = pd.DataFrame(rows)
    return pd.DataFrame({"n_checks": len(ts), "n_fallas": (diff > 0).sum(), "max_abs_diff": diff.max(),
                         "look_ahead": (diff > 0).any()})

In [ ]:
import lookahead; importlib.reload(lookahead)
la = lookahead.truncation_test(indicadores.add_indicators, raw, cols=IND_COLS, n_checks=40)
display(la)
print("¿Algún indicador con look-ahead?", la["look_ahead"].any())

# Control positivo: un indicador "tramposo" (media centrada) SÍ debe fallar
bad = lambda d: d["close"].rolling(21, center=True).mean().rename("sma_centrada")
display(lookahead.truncation_test(bad, raw, n_checks=20))

**Conclusión:** ninguno de nuestros indicadores cambia al truncar (todos son causales). El control positivo (media móvil *centrada*) sí cambia, lo que demuestra que la prueba detecta look-ahead cuando existe. Ojo: aunque el indicador sea causal, el look-ahead puede entrar después (p. ej. ejecutar al cierre de la misma barra que generó la señal); por eso en la Act 06 se prueba el pipeline completo.

# Act 05 — SPEC, entry rule, posición y unit tests
## Splits y parámetros base
Train / Test / Validation cronológicos (nunca se barajan). **Validation no se toca** hasta el final del notebook.

In [ ]:
TRAIN = ("2012-01-01", "2019-12-31")
TEST  = ("2020-01-01", "2022-12-31")
VALID = ("2023-01-01", "2025-12-31")

dfm = df.iloc[WARMUP:]                 # descartamos el warm-up de los indicadores
d_train, d_test, d_valid = (dfm.loc[a:b] for a, b in (TRAIN, TEST, VALID))
print({k: (len(v), v.index.min().date(), v.index.max().date()) for k, v in
       {"train": d_train, "test": d_test, "valid": d_valid}.items()})

# Parámetros base del SPEC
R_MULT, K_SL, RISK, MAX_HOLD = 2.0, 2.0, 0.01, 20
COMM_BPS, SLIP_BPS, BORROW = 0.5, 2.0, 0.0025
RT_BPS = 2 * (COMM_BPS + SLIP_BPS)                     # costo ida y vuelta
atr_pct = (d_train.atr_14 / d_train.close).median()
cost_R = (RT_BPS / 1e4) / (K_SL * atr_pct)             # costo expresado en unidades de R
p_star_gross = 1 / (1 + R_MULT)
p_star = (1 + cost_R) / (1 + R_MULT)
print(f"ATR/precio mediano (train) = {atr_pct:.3%}; distancia al stop = {K_SL*atr_pct:.3%}")
print(f"Costo ida y vuelta = {RT_BPS:.1f} bps = {cost_R:.4f} R")
print(f"Win rate break-even:  sin costos p* = 1/(1+r) = {p_star_gross:.2%}   |   con costos p* = {p_star:.2%}")

## 1. `SPEC.md` (se genera con los números calculados arriba)

In [ ]:
spec = f"""# SPEC — Estrategia Lab 02: Trend + Momentum con filtro RSI

## 1. Universe and frequency
- **Activo:** {TICKER} (ETF S&P 500, precios ajustados por dividendos y splits).
- **Frecuencia:** barras diarias OHLCV.
- **Rango:** {dfm.index.min().date()} → {dfm.index.max().date()} (tras {WARMUP} barras de warm-up).
- **Splits cronológicos:** Train {TRAIN[0]}→{TRAIN[1]} · Test {TEST[0]}→{TEST[1]} · Validation {VALID[0]}→{VALID[1]} (se usa una sola vez).

## 2. Features
| Indicador | Ventana | Familia |
|---|---|---|
| SMA rápida / lenta | 20 / 100 (grid: 10,20 / 50,100) | Tendencia |
| MACD histograma | EMA 12, EMA 26, señal 9 | Momentum |
| RSI (Wilder) | 14 | Momentum / oscilador |
| ATR (Wilder) | 14 | Volatilidad (solo para stops y sizing) |

## 3. Entry rule
$$long_t = \\mathbb{{1}}[SMA_{{20,t}} > SMA_{{100,t}}]\\cdot\\mathbb{{1}}[MACDh_t > 0]\\cdot\\mathbb{{1}}[RSI_{{14,t}} < 70]$$
$$short_t = \\mathbb{{1}}[SMA_{{20,t}} < SMA_{{100,t}}]\\cdot\\mathbb{{1}}[MACDh_t < 0]\\cdot\\mathbb{{1}}[RSI_{{14,t}} > 30]$$
$$s_t = long_t - short_t \\in \\{{-1,0,+1\\}}$$
Umbrales: signo de la diferencia de medias, 0 en MACDh, 70/30 en RSI. Se entra solo si se está plano y $s_t\\neq0$.

## 4. Exit rule
- **SL** = entrada − side·k·ATR₁₄, con k = {K_SL}. **TP** = entrada + side·r·k·ATR₁₄, con r = {R_MULT}.
- **Señal opuesta:** si $s_t = -side$ se cierra en la apertura siguiente; no se revierte en la misma barra.
- **Holding máximo:** {MAX_HOLD} barras, salida en la apertura siguiente.

## 5. Sizing
Fixed-fractional por riesgo: $acciones = \\dfrac{{equity_{{t-1}}\\cdot {RISK:.0%}}}{{|entrada - SL|}}$ → cada operación arriesga exactamente {RISK:.0%} del equity (1R). Acciones fraccionales permitidas.

## 6. Costs
| Concepto | Valor | Justificación |
|---|---|---|
| Comisión | {COMM_BPS} bps por lado | Broker de descuento (IBKR ≈ USD 0.0035–0.005/acción ≈ 0.1–0.2 bps en SPY), redondeado hacia arriba |
| Slippage | {SLIP_BPS} bps por lado | Spread de SPY ≈ 0.2–0.5 bps + impacto/ejecución en la apertura; margen conservador |
| Borrow fee | {BORROW:.2%} anual sobre cortos | SPY es *general collateral* (tasa de préstamo muy baja) |
| **Ida y vuelta** | **{RT_BPS:.1f} bps** | Sensibilidad 0–50 bps en la Act 06 |

## 7. Conventions
- La señal se calcula con el **cierre de t** y se ejecuta en la **apertura de t+1** (nunca al mismo cierre).
- SL/TP se fijan con el ATR de la barra de decisión.
- **Empates intrabarra:** si SL y TP caen en la misma barra se asume **SL** (pesimista). Si la barra abre más allá del SL/TP (gap), se llena a la apertura.
- Una sola posición a la vez.

## 8. Umbral de break-even
Con r = {R_MULT}: $p^* = \\dfrac{{1}}{{1+r}} = {p_star_gross:.2%}$ sin costos.
Incluyendo costos ({RT_BPS:.1f} bps ≈ {cost_R:.3f} R por operación): $p^* = \\dfrac{{1+c_R}}{{1+r}} = {p_star:.2%}$.
**La estrategia debe superar un win rate de {p_star:.2%}** (con el R medio realizado ≈ r).
"""
with open("SPEC.md", "w", encoding="utf-8") as f:
    f.write(spec)
from IPython.display import Markdown
Markdown(spec)

## 2. `src/strategy.py` — entry rule + estructura de posición

In [ ]:
%%writefile src/strategy.py
"""Entry rule del SPEC + estructura de posición + máquina de estados."""
from dataclasses import dataclass, field
from typing import Optional
import numpy as np
import pandas as pd


@dataclass
class Position:
    side: int              # +1 largo, -1 corto
    shares: float          # número de acciones (fraccionales permitidas)
    entry_price: float
    stop_loss: float
    take_profit: float
    entry_bar: int = 0
    entry_date: Optional[pd.Timestamp] = None
    costs: float = 0.0     # costos acumulados de la operación

    @property
    def risk_per_share(self):
        return abs(self.entry_price - self.stop_loss)


def combine_signals(df, fast=20, slow=100, rsi_lo=30, rsi_hi=70):
    """Entry rule:
        long_t  = 1[SMA_fast > SMA_slow] * 1[MACD_hist > 0] * 1[RSI_14 < rsi_hi]
        short_t = 1[SMA_fast < SMA_slow] * 1[MACD_hist < 0] * 1[RSI_14 > rsi_lo]
        s_t = long_t - short_t  ∈ {-1, 0, +1}
    Tendencia (SMA) + momentum (MACD) + filtro de agotamiento (RSI)."""
    trend = df[f"sma_{fast}"] - df[f"sma_{slow}"]
    h, r = df["macd_hist"], df["rsi_14"]
    long_ = (trend > 0) & (h > 0) & (r < rsi_hi)
    short = (trend < 0) & (h < 0) & (r > rsi_lo)
    return (long_.astype(int) - short.astype(int)).rename("signal")


def position_size(equity, risk_frac, entry_price, stop_loss):
    """Fixed-fractional: acciones tales que (entry - stop) * acciones = equity * risk_frac."""
    dist = abs(entry_price - stop_loss)
    if dist <= 0 or not np.isfinite(dist):
        return 0.0
    return equity * risk_frac / dist


def make_brackets(side, entry, atr, k_sl, r):
    stop = entry - side * k_sl * atr
    target = entry + side * r * k_sl * atr
    return stop, target


def check_exit(pos, o, h, l):
    """Salida intrabarra. Convención pesimista: si SL y TP caen en la misma barra -> stop_loss.
    Si la barra abre más allá del nivel (gap), se llena a la apertura."""
    if pos.side == 1:
        if o <= pos.stop_loss:
            return o, "stop_loss"
        if o >= pos.take_profit:
            return o, "take_profit"
        if l <= pos.stop_loss:
            return pos.stop_loss, "stop_loss"
        if h >= pos.take_profit:
            return pos.take_profit, "take_profit"
    else:
        if o >= pos.stop_loss:
            return o, "stop_loss"
        if o <= pos.take_profit:
            return o, "take_profit"
        if h >= pos.stop_loss:
            return pos.stop_loss, "stop_loss"
        if l <= pos.take_profit:
            return pos.take_profit, "take_profit"
    return None


class PositionBook:
    """Máquina de estados FLAT <-> OPEN. Nunca permite dos posiciones abiertas."""

    def __init__(self):
        self.position = None

    @property
    def is_flat(self):
        return self.position is None

    def open(self, pos):
        if self.position is not None:
            raise RuntimeError("Ya hay una posición abierta: no se permiten dos a la vez")
        self.position = pos

    def close(self):
        if self.position is None:
            raise RuntimeError("No hay posición que cerrar")
        p, self.position = self.position, None
        return p

In [ ]:
import strategy; importlib.reload(strategy)
sig = strategy.combine_signals(dfm)
print(sig.value_counts().rename({1: "largo", 0: "plano", -1: "corto"}))

## 3. Unit tests (a) SL+TP misma barra → `stop_loss`, (b) sizing exacto, (c) nunca dos posiciones

In [ ]:
%%writefile tests/conftest.py
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))

In [ ]:
%%writefile tests/test_strategy.py
import numpy as np
import pandas as pd
import pytest
from strategy import Position, PositionBook, check_exit, position_size
from backtest import backtest
from data_utils import synthetic_ohlcv
from indicadores import add_indicators


# (a) largo con SL y TP dentro de la misma barra -> stop_loss
def test_long_sl_and_tp_same_bar_is_stop_loss():
    pos = Position(side=1, shares=10, entry_price=100, stop_loss=98, take_profit=104)
    price, reason = check_exit(pos, o=100.5, h=105, l=97)   # toca 98 y 104
    assert reason == "stop_loss"
    assert price == 98


def test_short_sl_and_tp_same_bar_is_stop_loss():
    pos = Position(side=-1, shares=10, entry_price=100, stop_loss=102, take_profit=96)
    assert check_exit(pos, o=99.5, h=103, l=95)[1] == "stop_loss"


# (b) el sizing regresa exactamente el riesgo presupuestado
@pytest.mark.parametrize("equity,risk,entry,stop", [(100_000, 0.01, 50, 48.5), (25_000, 0.02, 412.3, 420.1),
                                                     (1_000, 0.005, 3.7, 3.2)])
def test_sizing_returns_exact_dollar_risk(equity, risk, entry, stop):
    shares = position_size(equity, risk, entry, stop)
    assert shares * abs(entry - stop) == pytest.approx(equity * risk, rel=1e-12)


# (c) la máquina de estados nunca mantiene dos posiciones abiertas
def test_book_rejects_second_position():
    book = PositionBook()
    book.open(Position(1, 1, 100, 99, 102))
    with pytest.raises(RuntimeError):
        book.open(Position(-1, 1, 100, 101, 98))


def test_backtest_never_overlaps_positions():
    df = add_indicators(synthetic_ohlcv(n=1200, seed=7))
    rng = np.random.default_rng(0)
    sig = pd.Series(rng.choice([-1, 0, 1], size=len(df)), index=df.index)   # señales caóticas
    res = backtest(df, sig, max_hold=5)
    tr = res["trades"].dropna(subset=["exit_bar"]).sort_values("entry_bar")
    assert len(tr) > 20
    assert (tr["entry_bar"].values[1:] > tr["exit_bar"].values[:-1]).all()
    assert (np.sign(res["position"]).abs() <= 1).all()

## 4. Win rate de break-even
Ya quedó declarado en `SPEC.md` (sección 8). Derivación: con probabilidad $p$ se gana $r$ R y con $1-p$ se pierde 1 R, menos el costo $c_R$:
$$E = p\,r - (1-p) - c_R = 0 \;\Rightarrow\; p^* = \frac{1+c_R}{1+r}$$

In [ ]:
rs = np.array([1, 1.5, 2, 3, 4])
pd.DataFrame({"r": rs, "p* sin costos": 1 / (1 + rs), "p* con costos": (1 + cost_R) / (1 + rs)}).set_index("r")

# Act 06 — Backtest orientado a eventos
## 1. `backtest()` como función pura con cash, posición y equity explícitos

In [ ]:
%%writefile src/backtest.py
"""Motor orientado a eventos (barra por barra), función pura: no modifica sus entradas."""
import numpy as np
import pandas as pd
from strategy import Position, PositionBook, position_size, make_brackets, check_exit

DEFAULTS = dict(k_sl=2.0, r=2.0, risk_frac=0.01, max_hold=20, commission_bps=0.5,
                slippage_bps=2.0, borrow_annual=0.0025, initial_cash=100_000.0)


def backtest(df, signal, k_sl=2.0, r=2.0, risk_frac=0.01, max_hold=20, commission_bps=0.5,
             slippage_bps=2.0, borrow_annual=0.0025, initial_cash=100_000.0):
    """Convenciones:
      * la señal s_t se calcula al cierre de t y se ejecuta en la APERTURA de t+1;
      * stops/targets se fijan con ATR_t de la barra de decisión;
      * SL y TP en la misma barra -> stop_loss; gaps -> se llena a la apertura;
      * señal opuesta -> se cierra en la siguiente apertura (no se revierte en la misma barra);
      * max_hold barras -> se cierra en la siguiente apertura.
    k_sl y r pueden ser escalares o arreglos por barra (Act 07)."""
    n = len(df)
    o, h, l, c = (df[x].to_numpy(float) for x in ("open", "high", "low", "close"))
    atr_ = df["atr_14"].to_numpy(float)
    sig = np.nan_to_num(np.asarray(signal, float))
    k_arr = np.broadcast_to(np.asarray(k_sl, float), (n,))
    r_arr = np.broadcast_to(np.asarray(r, float), (n,))
    comm, slip = commission_bps / 1e4, slippage_bps / 1e4
    idx = df.index

    cash = float(initial_cash)
    book = PositionBook()
    equity = np.empty(n); cash_s = np.empty(n); pos_s = np.zeros(n)
    trades = []
    pending_entry, pending_exit = 0, False
    prev_equity = cash

    def _close(t, raw_price, reason):
        nonlocal cash
        p = book.close()
        fill = raw_price * (1 - p.side * slip)          # slippage adverso
        fee = abs(p.shares * fill) * comm
        cash += p.side * p.shares * fill - fee
        p.costs += fee  # la slippage ya está implícita en los precios de llenado
        pnl = p.side * p.shares * (fill - p.entry_price) - p.costs
        trades.append(dict(side=p.side, entry_date=p.entry_date, entry_bar=p.entry_bar, entry_price=p.entry_price,
                           shares=p.shares, stop_loss=p.stop_loss, take_profit=p.take_profit,
                           exit_date=idx[t], exit_bar=t, exit_price=fill, reason=reason,
                           pnl=pnl, R=pnl / (p.risk_per_share * p.shares) if p.shares else 0.0))

    for t in range(n):
        # 1) ejecución en la apertura de lo decidido al cierre anterior
        if pending_exit and not book.is_flat:
            _close(t, o[t], "signal_exit" if pending_exit == "signal" else "max_hold")
        pending_exit = False
        if pending_entry != 0 and book.is_flat:
            side = int(pending_entry)
            fill = o[t] * (1 + side * slip)
            stop, target = make_brackets(side, fill, atr_[t - 1], k_arr[t - 1], r_arr[t - 1])
            shares = position_size(prev_equity, risk_frac, fill, stop)
            if shares > 0:
                fee = shares * fill * comm
                cash -= side * shares * fill + fee
                book.open(Position(side, shares, fill, stop, target, t, idx[t],
                                   costs=fee))
        pending_entry = 0

        # 2) salidas intrabarra por SL / TP
        if not book.is_flat:
            ex = check_exit(book.position, o[t], h[t], l[t])
            if ex is not None:
                _close(t, *ex)

        # 3) borrow fee de cortos (se cobra al cierre)
        if not book.is_flat and book.position.side == -1:
            fee = book.position.shares * c[t] * borrow_annual / 252
            cash -= fee; book.position.costs += fee

        # 4) mark-to-market
        p = book.position
        equity[t] = cash + (p.side * p.shares * c[t] if p else 0.0)
        cash_s[t] = cash; pos_s[t] = p.side * p.shares if p else 0.0
        prev_equity = equity[t]

        # 5) decisiones al cierre de t (se ejecutan en t+1)
        if p is not None:
            if sig[t] == -p.side:
                pending_exit = "signal"
            elif t - p.entry_bar + 1 >= max_hold:
                pending_exit = "max_hold"
        elif sig[t] != 0 and np.isfinite(atr_[t]) and atr_[t] > 0:
            pending_entry = int(sig[t])

    tr = pd.DataFrame(trades)
    if not book.is_flat:  # posición viva: se reporta pero NO altera el equity (evita romper el truncamiento)
        p = book.position
        tr = pd.concat([tr, pd.DataFrame([dict(side=p.side, entry_date=p.entry_date, entry_bar=p.entry_bar,
                                               entry_price=p.entry_price, shares=p.shares, stop_loss=p.stop_loss,
                                               take_profit=p.take_profit, exit_date=None, exit_bar=None,
                                               exit_price=c[-1], reason="open",
                                               pnl=p.side * p.shares * (c[-1] - p.entry_price) - p.costs,
                                               R=np.nan)])], ignore_index=True)
    return dict(equity=pd.Series(equity, idx, name="equity"), cash=pd.Series(cash_s, idx, name="cash"),
                position=pd.Series(pos_s, idx, name="position"), trades=tr)

In [ ]:
%%writefile src/metrics.py
import numpy as np
import pandas as pd

PPY = 252


def returns(eq):
    return eq.pct_change().fillna(0.0)


def cagr(eq, ppy=PPY):
    yrs = len(eq) / ppy
    return (eq.iloc[-1] / eq.iloc[0]) ** (1 / yrs) - 1 if yrs > 0 and eq.iloc[0] > 0 else np.nan


def ann_vol(eq, ppy=PPY):
    return returns(eq).std() * np.sqrt(ppy)


def sharpe(eq, rf=0.0, ppy=PPY):
    r = returns(eq) - rf / ppy
    return r.mean() / r.std() * np.sqrt(ppy) if r.std() > 0 else 0.0


def sortino(eq, ppy=PPY):
    r = returns(eq); dn = r[r < 0].std()
    return r.mean() / dn * np.sqrt(ppy) if dn and dn > 0 else 0.0


def drawdown(eq):
    return eq / eq.cummax() - 1


def max_drawdown(eq):
    return drawdown(eq).min()


def calmar(eq, ppy=PPY):
    mdd = abs(max_drawdown(eq))
    return cagr(eq, ppy) / mdd if mdd > 0 else np.nan


def turnover(trades, eq, ppy=PPY):
    """Nocional operado (entrada+salida) / equity medio, anualizado."""
    if trades is None or len(trades) == 0:
        return 0.0
    closed = trades[trades["reason"] != "open"]
    notional = (closed["shares"].abs() * (closed["entry_price"] + closed["exit_price"])).sum()
    return notional / eq.mean() / (len(eq) / ppy)


def win_rate(trades):
    closed = trades[trades["reason"] != "open"] if len(trades) else trades
    return (closed["pnl"] > 0).mean() if len(closed) else np.nan


def breakeven_winrate(r, cost_R=0.0):
    """p* tal que p*r - (1-p) - cost_R = 0  ->  p* = (1 + cost_R) / (1 + r)."""
    return (1 + cost_R) / (1 + r)


def payoff_breakeven(closed):
    """p* usando el payoff REALIZADO: |pérdida media| / (ganancia media + |pérdida media|)."""
    if len(closed) == 0:
        return np.nan
    w, l = closed.loc[closed["pnl"] > 0, "pnl"].mean(), -closed.loc[closed["pnl"] <= 0, "pnl"].mean()
    return l / (w + l) if np.isfinite(w) and np.isfinite(l) and (w + l) > 0 else np.nan


def summary(res, r=2.0, cost_R=0.0, name=None):
    eq, tr = res["equity"], res["trades"]
    closed = tr[tr["reason"] != "open"] if len(tr) else tr
    return pd.Series({"CAGR": cagr(eq), "Vol": ann_vol(eq), "Sharpe": sharpe(eq), "Sortino": sortino(eq),
                      "MaxDD": max_drawdown(eq), "Calmar": calmar(eq), "Turnover": turnover(tr, eq),
                      "Trades": len(closed), "WinRate_emp": win_rate(tr),
                      "p*_teórico": breakeven_winrate(r, cost_R),
                      "R_medio": closed["R"].mean() if len(closed) else np.nan,
                      "p*_payoff_real": payoff_breakeven(closed),
                      "Equity_final": eq.iloc[-1]}, name=name)

In [ ]:
%%writefile src/pipeline.py
import pandas as pd
from indicadores import add_indicators
from strategy import combine_signals
from backtest import backtest


def run_pipeline(df_raw, theta=None, **bt_kwargs):
    """indicadores -> señales -> combinación -> backtest."""
    theta = theta or dict(fast=20, slow=100, k_sl=2.0)
    d = add_indicators(df_raw)
    sig = combine_signals(d, fast=theta["fast"], slow=theta["slow"])
    res = backtest(d, sig, k_sl=theta["k_sl"], **bt_kwargs)
    return pd.DataFrame({"signal": sig, "equity": res["equity"], "position": res["position"]})

## 2. Golden-file test y prueba de truncamiento del pipeline completo
El equity esperado del golden file está calculado a mano en el docstring (no corriendo el código).

In [ ]:
%%writefile tests/test_backtest.py
import numpy as np
import pandas as pd
import pytest
from backtest import backtest
from data_utils import synthetic_ohlcv
from pipeline import run_pipeline


def test_golden_file():
    """Calculado a mano (sin costos, k_sl=1, r=2, riesgo 1%, capital 10,000):
    t0: cierre, señal +1 -> entrar en la apertura de t1.        equity = 10,000
    t1: entra a 100. ATR_0 = 2 -> SL = 98, TP = 100 + 2*1*2 = 104.
        acciones = 10,000*0.01 / (100-98) = 50. cash = 10,000 - 50*100 = 5,000.
        cierre 100.5 -> equity = 5,000 + 50*100.5 = 10,025
    t2: H=103 < 104, L=100 > 98. cierre 102 -> equity = 5,000 + 5,100 = 10,100
    t3: H=105 >= 104 -> take-profit en 104. cash = 5,000 + 50*104 = 10,200 -> equity 10,200
    t4: plano -> equity 10,200.   Ganancia = 50*(104-100) = 200 = 2R ✔"""
    idx = pd.bdate_range("2024-01-01", periods=5)
    df = pd.DataFrame({"open": [99, 100, 101, 102, 104.5], "high": [100, 101, 103, 105, 105],
                       "low": [98, 99, 100, 101, 104], "close": [99.5, 100.5, 102, 104.5, 104.8],
                       "atr_14": [2, 2, 2, 2, 2]}, index=idx)
    sig = [1, 0, 0, 0, 0]
    res = backtest(df, sig, k_sl=1.0, r=2.0, risk_frac=0.01, max_hold=100, commission_bps=0,
                   slippage_bps=0, borrow_annual=0, initial_cash=10_000)
    expected = [10_000, 10_025, 10_100, 10_200, 10_200]
    assert res["equity"].tolist() == pytest.approx(expected)
    assert res["trades"].iloc[0]["reason"] == "take_profit"


def test_golden_file_short_stop():
    """Corto, SL y TP en la misma barra -> stop. k=1, r=2, ATR=1, capital 10,000, riesgo 1%.
    t1: vende a 50, SL = 51, TP = 48, acciones = 100/1 = 100, cash = 10,000 + 5,000 = 15,000.
        cierre 50 -> equity = 15,000 - 100*50 = 10,000
    t2: H=51.5 y L=47.5 (ambos) -> stop en 51: cash = 15,000 - 5,100 = 9,900 -> equity 9,900 (=-1R)"""
    idx = pd.bdate_range("2024-01-01", periods=3)
    df = pd.DataFrame({"open": [50, 50, 50], "high": [50.5, 50.5, 51.5], "low": [49.5, 49.5, 47.5],
                       "close": [50, 50, 49], "atr_14": [1, 1, 1]}, index=idx)
    res = backtest(df, [-1, 0, 0], k_sl=1, r=2, commission_bps=0, slippage_bps=0, borrow_annual=0,
                   initial_cash=10_000)
    assert res["equity"].tolist() == pytest.approx([10_000, 10_000, 9_900])


def test_pipeline_truncation():
    """indicadores -> señales -> combinación -> backtest: el valor en t no cambia al truncar."""
    raw = synthetic_ohlcv(n=900, seed=3)
    full = run_pipeline(raw)
    rng = np.random.default_rng(1)
    for t in sorted(rng.choice(np.arange(150, len(raw)), 12, replace=False)):
        part = run_pipeline(raw.iloc[:t + 1])
        assert part["signal"].iloc[-1] == full["signal"].iloc[t]
        assert part["equity"].iloc[-1] == pytest.approx(full["equity"].iloc[t], rel=1e-12)
        assert part["position"].iloc[-1] == pytest.approx(full["position"].iloc[t], rel=1e-12)

In [ ]:
!python -m pytest -q tests

Además, la prueba de truncamiento del pipeline sobre los **datos reales**:

In [ ]:
import backtest as bt, metrics as mt, pipeline; importlib.reload(bt); importlib.reload(mt); importlib.reload(pipeline)
pipe = lambda d: pipeline.run_pipeline(d)
lookahead.truncation_test(pipe, raw.loc[:TEST[1]], n_checks=15, warmup=300)

## 3. Selección de θ* en Train
Grid pequeño (12 combinaciones, para limitar el data snooping). Objetivo $J$ = Sharpe en train.

In [ ]:
BT_KW = dict(r=R_MULT, risk_frac=RISK, max_hold=MAX_HOLD, commission_bps=COMM_BPS,
             slippage_bps=SLIP_BPS, borrow_annual=BORROW)
GRID = [dict(fast=f, slow=s, k_sl=k) for f in (10, 20) for s in (50, 100) for k in (1.5, 2.0, 3.0)]

def run(d, theta, mask=None, **kw):
    s = strategy.combine_signals(d, theta["fast"], theta["slow"])
    if mask is not None:
        s = s.where(mask, 0)
    return bt.backtest(d, s, k_sl=theta["k_sl"], **{**BT_KW, **kw})

grid_res = pd.DataFrame([{**th, "Sharpe_train": mt.sharpe(run(d_train, th)["equity"])} for th in GRID])
THETA_STAR = grid_res.sort_values("Sharpe_train", ascending=False).iloc[0][["fast", "slow", "k_sl"]].to_dict()
THETA_STAR = dict(fast=int(THETA_STAR["fast"]), slow=int(THETA_STAR["slow"]), k_sl=float(THETA_STAR["k_sl"]))
display(grid_res.sort_values("Sharpe_train", ascending=False))
print("θ* =", THETA_STAR)

## 4. Curva de equity, drawdown y lista de operaciones

In [ ]:
res_tr, res_te = run(d_train, THETA_STAR), run(d_test, THETA_STAR)
bh = lambda d: 100_000 * d.close / d.close.iloc[0]

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex="col", gridspec_kw={"height_ratios": [2, 1]})
for j, (name, res, d) in enumerate([("Train", res_tr, d_train), ("Test", res_te, d_test)]):
    axes[0, j].plot(res["equity"], label="Estrategia"); axes[0, j].plot(bh(d), alpha=.6, label="Buy & Hold")
    axes[0, j].set_title(f"Equity — {name}"); axes[0, j].legend(); axes[0, j].grid(alpha=.3)
    axes[1, j].fill_between(res["equity"].index, mt.drawdown(res["equity"]), color="red", alpha=.4)
    axes[1, j].set_title(f"Drawdown — {name}"); axes[1, j].grid(alpha=.3)
fig.tight_layout(); fig.savefig("figures/equity_drawdown.png"); plt.show()

trades = pd.concat([res_tr["trades"].assign(split="train"), res_te["trades"].assign(split="test")])
trades.to_csv("figures/trades.csv", index=False)
display(trades.head(15))
display(trades.groupby(["split", "reason"]).size().unstack(fill_value=0))

## 5. Sensibilidad a costos (costo ida y vuelta de 0 a 50 bps)

In [ ]:
rows = []
for rt in range(0, 55, 5):
    kw = dict(commission_bps=0.0, slippage_bps=rt / 2)
    for name, d in (("train", d_train), ("test", d_test)):
        e = run(d, THETA_STAR, **kw)["equity"]
        rows.append(dict(rt_bps=rt, split=name, Sharpe=mt.sharpe(e), Equity_final=e.iloc[-1]))
sens = pd.DataFrame(rows)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name, g in sens.groupby("split"):
    ax[0].plot(g.rt_bps, g.Sharpe, marker="o", label=name)
    ax[1].plot(g.rt_bps, g.Equity_final, marker="o", label=name)
ax[0].axhline(0, c="k", lw=.7); ax[0].axvline(RT_BPS, ls="--", c="gray", label="costo SPEC")
ax[1].axhline(100_000, c="k", lw=.7)
ax[0].set(title="Sharpe vs costo ida y vuelta", xlabel="bps"); ax[1].set(title="Equity final vs costo", xlabel="bps")
for a in ax: a.legend(); a.grid(alpha=.3)
fig.tight_layout(); fig.savefig("figures/sensibilidad_costos.png"); plt.show()
be = sens[(sens.split == "train") & (sens.Sharpe <= 0)].rt_bps.min()
print(f"En train el Sharpe se vuelve ≤ 0 a partir de ~{be} bps ida y vuelta." if pd.notna(be)
      else "En train el Sharpe se mantiene positivo hasta 50 bps.")

## 6. Métricas: desempeño, MaxDD, Calmar, turnover, win rate teórico vs empírico

In [ ]:
def bh_res(d):
    return dict(equity=bh(d), trades=pd.DataFrame(columns=["reason", "pnl", "R", "shares", "entry_price", "exit_price"]))

metrics_06 = pd.concat([mt.summary(res_tr, R_MULT, cost_R, "Estrategia train"),
                        mt.summary(res_te, R_MULT, cost_R, "Estrategia test"),
                        mt.summary(bh_res(d_train), R_MULT, cost_R, "B&H train"),
                        mt.summary(bh_res(d_test), R_MULT, cost_R, "B&H test")], axis=1)
display(metrics_06)
for s in ("train", "test"):
    m = metrics_06[f"Estrategia {s}"]
    v1 = "SUPERA" if m.WinRate_emp > m["p*_teórico"] else "NO supera"
    v2 = "SUPERA" if m.WinRate_emp > m["p*_payoff_real"] else "NO supera"
    print(f"{s}: win rate empírico {m.WinRate_emp:.1%} | p* teórico (SPEC, r={R_MULT}) {m['p*_teórico']:.1%} → {v1} | "
          f"p* con el payoff realizado {m['p*_payoff_real']:.1%} → {v2}")
print('''
Nota: el p* del SPEC supone que cada ganadora paga r·R. Si muchas operaciones salen por señal opuesta o
por holding máximo antes de tocar el TP, el payoff real es menor que r y el umbral efectivo sube; por eso
se reportan ambos. Lo que importa es superar el p* con el payoff realizado.''')

## 7. Auditoría de sesgos

In [ ]:
la_pipe = lookahead.truncation_test(pipe, raw.loc[:TEST[1]], n_checks=10, warmup=300)
deg = metrics_06.loc["Sharpe", "Estrategia train"] - metrics_06.loc["Sharpe", "Estrategia test"]
share_top = (grid_res.Sharpe_train >= grid_res.Sharpe_train.max() * 0.8).mean()
ties = res_tr["trades"].query("reason=='stop_loss'")
audit = pd.DataFrame([
    ("Look-ahead", f"Prueba de truncamiento del pipeline completo: {int(la_pipe.n_fallas.sum())} fallas en "
                   f"{int(la_pipe.n_checks.iloc[0])} cortes; señal al cierre t, ejecución en apertura t+1.", "Controlado"),
    ("Survivorship", f"Un solo activo ({TICKER}) que existió todo el periodo; pero se eligió ex-post un índice "
                     "de EE.UU. que sobrevivió y ganó. No hay universo de acciones deslistadas.", "Riesgo residual"),
    ("Overfitting / data snooping", f"Grid de {len(GRID)} combinaciones optimizado solo en train. Sharpe train−test = {deg:.2f}. "
                                     f"{share_top:.0%} del grid queda a ≥80% del mejor (superficie {'plana' if share_top>.3 else 'puntiaguda'}). "
                                     "Validation reservada y usada una sola vez.", "Mitigado"),
    ("Ejecución optimista", f"Llenado en apertura t+1 con {SLIP_BPS} bps de slippage + {COMM_BPS} bps comisión; "
                            f"SL gana los empates intrabarra; gaps llenan a la apertura. {len(ties)} stops en train.", "Conservador"),
    ("Selección / periodo de muestra", f"{dfm.index.min().year}–{dfm.index.max().year} es mayormente alcista "
                                       f"(B&H train CAGR {metrics_06.loc['CAGR','B&H train']:.1%}); test incluye COVID-2020 y bajista 2022. "
                                       "Resultados dependen del régimen → motivación de la Act 07.", "Riesgo residual"),
], columns=["Sesgo", "Evidencia concreta", "Estado"]).set_index("Sesgo")
with pd.option_context("display.max_colwidth", None):
    display(audit)

# Act 07 — Regime Detection
## 1. `regime_features()` + prueba de truncamiento en cada columna

In [ ]:
%%writefile src/regimes.py
"""Act 07: features de régimen y clasificadores (reglas, K-means, HMM)."""
import numpy as np
import pandas as pd
from scipy.stats import multivariate_normal

REGIME_NAMES = {0: "Alcista", 1: "Lateral", 2: "Estrés"}


def regime_features(df):
    """Todas causales (rolling hacia atrás)."""
    c = df["close"]; lr = np.log(c).diff()
    f = pd.DataFrame(index=df.index)
    f["ret_20"] = np.log(c / c.shift(20))                         # retorno acumulado 1 mes
    f["vol_20"] = lr.rolling(20).std() * np.sqrt(252)             # vol realizada anualizada
    f["trend_100"] = c / c.rolling(100).mean() - 1                # distancia a la SMA100
    f["vol_ratio"] = lr.rolling(20).std() / lr.rolling(100).std() # vol corta vs larga
    f["dd_252"] = c / c.rolling(252, min_periods=1).max() - 1     # drawdown desde máximo 1 año
    return f


def rules_classifier(F, vol_q):
    """Reglas: Estrés si vol_20 > q75(train); si no, Alcista si trend_100>0, si no Lateral."""
    lab = np.where(F["trend_100"] > 0, 0, 1)
    lab = np.where(F["vol_20"] > vol_q, 2, lab)
    return pd.Series(lab, F.index, name="rules")


def name_states(F_train, raw_labels):
    """Mapea etiquetas arbitrarias a {0 Alcista, 1 Lateral, 2 Estrés} usando centroides:
    el de mayor vol_20 = Estrés; de los otros, mayor trend_100 = Alcista."""
    cent = F_train.groupby(raw_labels).mean()
    stress = cent["vol_20"].idxmax()
    rest = cent.drop(stress)
    bull = rest["trend_100"].idxmax()
    mapping = {stress: 2, bull: 0}
    for s in cent.index:
        mapping.setdefault(s, 1)
    return mapping


def hmm_filter(model, X):
    """Algoritmo forward: P(s_t | x_1..x_t). Solo pasado -> etiqueta FILTRADA (sin look-ahead)."""
    K, T = model.n_components, len(X)
    covs = model.covars_
    logB = np.column_stack([multivariate_normal(model.means_[k], covs[k], allow_singular=True).logpdf(X)
                            for k in range(K)])
    alpha = np.zeros((T, K))
    a = np.log(model.startprob_ + 1e-300) + logB[0]
    alpha[0] = np.exp(a - a.max()); alpha[0] /= alpha[0].sum()
    A = model.transmat_
    for t in range(1, T):
        prior = alpha[t - 1] @ A
        a = np.log(prior + 1e-300) + logB[t]
        alpha[t] = np.exp(a - a.max()); alpha[t] /= alpha[t].sum()
    return alpha


def run_lengths(labels):
    lab = np.asarray(labels)
    changes = np.r_[True, lab[1:] != lab[:-1]]
    ids = np.cumsum(changes)
    return pd.Series(ids).value_counts().values


def regime_table(labels, X_scaled=None, index=None):
    from sklearn.metrics import silhouette_score
    lab = pd.Series(np.asarray(labels), index=index)
    months = (lab.index[-1] - lab.index[0]).days / 30.44
    out = {"silhouette": silhouette_score(X_scaled, lab.values, sample_size=min(3000, len(lab)), random_state=0)
           if X_scaled is not None and lab.nunique() > 1 else np.nan,
           "duración_media (barras)": run_lengths(lab).mean(),
           "transiciones/mes": (lab.values[1:] != lab.values[:-1]).sum() / months}
    for k, v in REGIME_NAMES.items():
        out[f"% {v}"] = (lab == k).mean()
    return pd.Series(out)

In [ ]:
import regimes; importlib.reload(regimes)
F_all = regimes.regime_features(raw)
FEATS = list(F_all.columns)
display(lookahead.truncation_test(regimes.regime_features, raw, n_checks=40, warmup=300))
F = F_all.loc[dfm.index].dropna()
F_train, F_test = F.loc[TRAIN[0]:TRAIN[1]], F.loc[TEST[0]:TEST[1]]
print("Todas las features pasan la prueba de truncamiento (look_ahead = False).")

## 2. Ajuste de los tres clasificadores **solo en train**
- **Reglas:** umbral de vol = percentil 75 de `vol_20` en train.
- **K-means (k=3):** sobre features estandarizadas (scaler ajustado en train).
- **HMM gaussiano (3 estados, covarianza completa):** la etiqueta que se opera es la **filtrada** $P(s_t\mid x_{1:t})$ (algoritmo forward, implementado en `regimes.hmm_filter`); Viterbi usa toda la muestra ⇒ solo sirve para comparar.

Todas las etiquetas se renombran a {0 Alcista, 1 Lateral, 2 Estrés} con los **centroides de train**.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from hmmlearn.hmm import GaussianHMM

scaler = StandardScaler().fit(F_train)
Z, Z_train = scaler.transform(F), scaler.transform(F_train)

# Reglas
VOL_Q = F_train.vol_20.quantile(0.75)
lab_rules = regimes.rules_classifier(F, VOL_Q)

# K-means
km = KMeans(3, n_init=20, random_state=0).fit(Z_train)
km_raw = pd.Series(km.predict(Z), F.index)
lab_km = km_raw.map(regimes.name_states(F_train, km_raw.loc[F_train.index]))

# HMM
hmm = GaussianHMM(3, covariance_type="full", n_iter=300, random_state=0).fit(Z_train)
alpha = regimes.hmm_filter(hmm, Z)                              # filtrado (causal)
hmm_filt_raw = pd.Series(alpha.argmax(1), F.index)
hmm_map = regimes.name_states(F_train, hmm_filt_raw.loc[F_train.index])
lab_hmm = hmm_filt_raw.map(hmm_map)
lab_vit = pd.Series(hmm.predict(Z), F.index).map(hmm_map)      # Viterbi (mira al futuro)
P_filt = pd.DataFrame(alpha, F.index).rename(columns=hmm_map).sort_index(axis=1)

LABELS = {"Reglas": lab_rules, "K-means": lab_km, "HMM filtrado": lab_hmm, "HMM Viterbi*": lab_vit}
print("Matriz de transición HMM (renombrada):")
order = [k for k, v in sorted(hmm_map.items(), key=lambda x: x[1])]
display(pd.DataFrame(hmm.transmat_[np.ix_(order, order)], index=list(regimes.REGIME_NAMES.values()),
                     columns=list(regimes.REGIME_NAMES.values())))

In [ ]:
# La etiqueta filtrada del HMM también debe pasar la prueba de truncamiento
def hmm_label_pipeline(d):
    f = regimes.regime_features(d).loc[lambda x: x.index >= F.index[0]].dropna()
    return pd.Series(regimes.hmm_filter(hmm, scaler.transform(f)).argmax(1), f.index, name="hmm_filtrado").astype(float)
display(lookahead.truncation_test(hmm_label_pipeline, raw, n_checks=10, warmup=600))

## 3. Tabla comparativa

In [ ]:
tables = {}
for split, idx, Zs in (("train", F_train.index, Z_train), ("test", F_test.index, scaler.transform(F_test))):
    tables[split] = pd.DataFrame({k: regimes.regime_table(v.loc[idx], Zs, idx) for k, v in LABELS.items()})
comp = pd.concat(tables, axis=1)
display(comp)
print("* Viterbi usa información futura: se reporta solo como referencia, no es operable.")

## 4. Línea de tiempo del precio coloreada por la etiqueta **filtrada** (y Viterbi al lado)

In [ ]:
COLORS = {0: "tab:green", 1: "tab:orange", 2: "tab:red"}
def color_timeline(ax, labels, title):
    px = dfm.close.loc[labels.index]
    ax.plot(px.index, px, c="lightgray", lw=.8, zorder=1)
    for k, name in regimes.REGIME_NAMES.items():
        m = labels == k
        ax.scatter(px.index[m], px[m], s=3, c=COLORS[k], label=name, zorder=2)
    for x in (TEST[0], VALID[0]):
        ax.axvline(pd.Timestamp(x), c="k", ls="--", lw=.8)
    ax.set_yscale("log"); ax.set_title(title); ax.legend(markerscale=4, loc="upper left"); ax.grid(alpha=.3)

fig, axes = plt.subplots(4, 1, figsize=(14, 15), sharex=True)
color_timeline(axes[0], lab_hmm, "HMM — etiqueta FILTRADA (la que se opera)")
color_timeline(axes[1], lab_vit, "HMM — Viterbi (suavizada, usa el futuro)")
color_timeline(axes[2], lab_km, "K-means")
color_timeline(axes[3], lab_rules, "Reglas")
fig.tight_layout(); fig.savefig("figures/regimenes_timeline.png"); plt.show()

agree = (lab_hmm == lab_vit).mean()
print(f"Filtrada y Viterbi coinciden en {agree:.1%} de las barras. La diferencia es el costo de no ver el futuro: "
      "la filtrada detecta los cambios con retraso y parpadea más en las transiciones.")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
ax.stackplot(P_filt.index, P_filt.T.values, colors=[COLORS[k] for k in P_filt.columns],
             labels=[regimes.REGIME_NAMES[k] for k in P_filt.columns], alpha=.8)
ax.set_title("Probabilidades filtradas del HMM  P(s_t | x_1..t)"); ax.legend(loc="upper left"); ax.set_ylim(0, 1)
plt.show()

## 5. Distribución de las features por régimen y nombres a partir de los centroides

In [ ]:
cent = pd.concat({k: F_train.groupby(v.loc[F_train.index]).mean().rename(index=regimes.REGIME_NAMES)
                  for k, v in LABELS.items() if k != "HMM Viterbi*"})
display(cent)

fig, axes = plt.subplots(1, len(FEATS), figsize=(18, 4))
for ax, f in zip(axes, FEATS):
    data = [F_train[f][lab_hmm.loc[F_train.index] == k] for k in regimes.REGIME_NAMES]
    bp = ax.boxplot(data, labels=list(regimes.REGIME_NAMES.values()), showfliers=False, patch_artist=True)
    for patch, k in zip(bp["boxes"], regimes.REGIME_NAMES):
        patch.set_facecolor(COLORS[k]); patch.set_alpha(.5)
    ax.set_title(f); ax.grid(alpha=.3)
fig.suptitle("Features por régimen (HMM filtrado, train)"); fig.tight_layout(); plt.show()

c = cent.loc["HMM filtrado"]
print(f'''Justificación de nombres (centroides HMM en train):
• Alcista: trend_100 = {c.loc["Alcista","trend_100"]:+.2%}, ret_20 = {c.loc["Alcista","ret_20"]:+.2%}, vol_20 = {c.loc["Alcista","vol_20"]:.1%} → precio sobre su SMA100, retornos positivos y vol baja.
• Lateral: trend_100 = {c.loc["Lateral","trend_100"]:+.2%}, ret_20 = {c.loc["Lateral","ret_20"]:+.2%}, vol_20 = {c.loc["Lateral","vol_20"]:.1%} → cerca de su media, retornos ~0, vol intermedia.
• Estrés: vol_20 = {c.loc["Estrés","vol_20"]:.1%}, vol_ratio = {c.loc["Estrés","vol_ratio"]:.2f}, dd_252 = {c.loc["Estrés","dd_252"]:.1%} → vol más alta, vol corta > larga y en drawdown.''')

## 6. Elección del método

In [ ]:
tr = comp["train"]
print(f'''Criterios (train):
                   silhouette   duración media   transiciones/mes
  Reglas           {tr.loc["silhouette","Reglas"]:8.3f}   {tr.loc["duración_media (barras)","Reglas"]:10.1f}   {tr.loc["transiciones/mes","Reglas"]:10.2f}
  K-means          {tr.loc["silhouette","K-means"]:8.3f}   {tr.loc["duración_media (barras)","K-means"]:10.1f}   {tr.loc["transiciones/mes","K-means"]:10.2f}
  HMM filtrado     {tr.loc["silhouette","HMM filtrado"]:8.3f}   {tr.loc["duración_media (barras)","HMM filtrado"]:10.1f}   {tr.loc["transiciones/mes","HMM filtrado"]:10.2f}''')

**Elijo el HMM con etiqueta filtrada.** Justificación:

1. **Es causal y operable.** La etiqueta filtrada solo usa $x_{1:t}$ (lo comprueba la prueba de truncamiento); Viterbi y el suavizado *forward-backward* no lo son.
2. **Modela la persistencia.** La matriz de transición tiene diagonal alta, así que penaliza cambiar de estado; K-means clasifica cada barra de forma independiente y "parpadea" más (más transiciones/mes ⇒ más cambios de θ ⇒ más turnover y costos).
3. **Da probabilidades**, no solo etiquetas: permite umbrales de confianza o ponderar θ por $P(s_t=j)$.
4. **Silhouette no es el criterio decisivo:** K-means optimiza justamente distancias euclidianas, así que casi siempre gana en silhouette; para trading importa más la duración de los regímenes y que sean estables fuera de muestra.
5. **Reglas** son transparentes y robustas, pero sus umbrales son arbitrarios y solo separan por vol y tendencia; quedan como *baseline*.

Contra: el HMM es sensible a la inicialización (fijé `random_state` y estandaricé) y a que los retornos no son gaussianos. *(Si en tus números K-means o reglas salen claramente mejor en duración y estabilidad de test, ajusta este párrafo.)*

## 7. Operar los regímenes: un θ por estado
$$\theta_j^* = \arg\max_\theta J\big(\text{backtest}(\text{train}\mid s_t=j,\theta)\big),\quad j=1,\dots,k$$
Para cada estado se hace el backtest de train permitiendo **entradas solo cuando la etiqueta filtrada es $j$**. Luego, barra a barra, se opera con $\theta^*_{\hat s_t}$ (señal y k del estado filtrado en $t$) y se compara contra el θ* único de la Act 06.

In [ ]:
state = lab_hmm.reindex(dfm.index)                  # etiqueta filtrada en t (conocida al cierre de t)
MIN_TRADES = 5
theta_state, rows = {}, []
for j, name in regimes.REGIME_NAMES.items():
    mask = (state.loc[d_train.index] == j)
    best, best_J = None, -np.inf
    for th in GRID:
        res = run(d_train, th, mask=mask)
        n_tr = (res["trades"]["reason"] != "open").sum() if len(res["trades"]) else 0
        J = mt.sharpe(res["equity"])
        rows.append({"estado": name, **th, "J_sharpe": J, "trades": n_tr})
        if n_tr >= MIN_TRADES and J > best_J:
            best, best_J = th, J
    theta_state[j] = best or THETA_STAR               # fallback si el estado tiene muy pocas operaciones
    print(f"θ*_{name:8s} = {theta_state[j]}   (J train = {best_J:.2f})")
grid_state = pd.DataFrame(rows)

In [ ]:
def run_regime(d):
    s_hat = state.loc[d.index].fillna(-1).astype(int)
    sig = pd.Series(0, d.index)
    k_arr = pd.Series(THETA_STAR["k_sl"], d.index, dtype=float)
    for j, th in theta_state.items():
        m = s_hat == j
        sig[m] = strategy.combine_signals(d, th["fast"], th["slow"])[m]
        k_arr[m] = th["k_sl"]
    return bt.backtest(d, sig, k_sl=k_arr.values, **BT_KW)

res_reg = {"train": run_regime(d_train), "test": run_regime(d_test)}
res_one = {"train": res_tr, "test": res_te}
cmp07 = pd.concat({s: pd.concat([mt.summary(res_one[s], R_MULT, cost_R, "θ* único (Act 06)"),
                                 mt.summary(res_reg[s], R_MULT, cost_R, "θ* por régimen")], axis=1)
                   for s in ("train", "test")}, axis=1)
display(cmp07)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, s, d in zip(axes, ("train", "test"), (d_train, d_test)):
    ax.plot(res_one[s]["equity"], label="θ* único"); ax.plot(res_reg[s]["equity"], label="θ* por régimen")
    ax.plot(bh(d), alpha=.5, label="Buy & Hold"); ax.set_title(s.capitalize()); ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig("figures/regimen_vs_unico.png"); plt.show()

In [ ]:
for s in ("train", "test"):
    a, b = cmp07[s]["θ* único (Act 06)"], cmp07[s]["θ* por régimen"]
    print(f"{s.upper():5s} Sharpe {a.Sharpe:.2f} → {b.Sharpe:.2f} | CAGR {a.CAGR:.1%} → {b.CAGR:.1%} | "
          f"MaxDD {a.MaxDD:.1%} → {b.MaxDD:.1%} | trades {a.Trades:.0f} → {b.Trades:.0f}")
gain_tr = cmp07["train"].loc["Sharpe"].diff().iloc[-1]; gain_te = cmp07["test"].loc["Sharpe"].diff().iloc[-1]
print(f'''
Lectura: en train el modelo por régimen {"mejora" if gain_tr>0 else "no mejora"} el Sharpe ({gain_tr:+.2f}); era de esperarse que
mejore porque tiene k veces más parámetros (más grados de libertad = más riesgo de overfitting). La prueba real es
test: ahí {"se mantiene la mejora" if gain_te>0 else "la mejora desaparece"} ({gain_te:+.2f}). {"Esto sugiere que el régimen aporta información útil." if gain_te>0 else "Esto indica que la ganancia en train era overfitting o que los regímenes de test difieren de los de train; me quedaría con el θ* único."}''')

## Validación final (una sola vez)
Se corre al final, sin volver a optimizar nada.

In [ ]:
val = pd.concat([mt.summary(run(d_valid, THETA_STAR), R_MULT, cost_R, "θ* único"),
                 mt.summary(run_regime(d_valid), R_MULT, cost_R, "θ* por régimen"),
                 mt.summary(bh_res(d_valid), R_MULT, cost_R, "Buy & Hold")], axis=1)
val

## Checklist de entrega
- `SPEC.md`, `src/{indicadores,plots,lookahead,strategy,backtest,metrics,pipeline,regimes,data_utils}.py`, `tests/`, `figures/` se generan al correr el notebook.
- Sube la carpeta a GitHub (incluye este notebook **ejecutado**) y entrega la URL del repo.
- `python -m pytest -q tests` debe pasar en verde.